# 6. Inspección inicial y limpieza de `TotalCharges`

Se revisa el CSV original y se genera `telco_step1.csv`, que sirve de entrada para la adecuación de tipos y el análisis de valores atípicos.

### Carga e inspección

Buscamos el archivo original desde el directorio de ejecución y sus padres. Revisamos dimensiones, tipos, valores presentes, estadísticas descriptivas y duplicados para conocer el estado inicial del dataset.

In [ ]:
from pathlib import Path

import pandas as pd

ruta_relativa = Path("data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
directorio = Path.cwd()
ruta_original = next(
    (base / ruta_relativa for base in (directorio, *directorio.parents)
     if (base / ruta_relativa).is_file()),
    None,
)
if ruta_original is None:
    raise FileNotFoundError(f"Abrí el notebook dentro del repositorio que contiene {ruta_relativa}.")

df = pd.read_csv(ruta_original)
print(f"Dimensiones: {df.shape[0]} filas y {df.shape[1]} columnas")
df.info()
print(df.describe(include="all").T.to_string())
print(f"Filas duplicadas: {df.duplicated().sum()}")
print(f"IDs de cliente repetidos: {df['customerID'].duplicated().sum()}")

El archivo contiene 7.043 registros y 21 variables. La inspección de tipos muestra que Pandas lee `TotalCharges` como texto. Para analizar la facturación acumulada, primero hay que identificar los valores vacíos y convertir la columna a numérica.

### Conversión y diagnóstico de faltantes

Quitamos espacios de los extremos, convertimos las cadenas vacías a faltantes y aplicamos `to_numeric`. La conversión genera un error si encuentra texto inesperado, para que el tratamiento de vacíos no oculte un dato inválido. Luego revisamos la antigüedad y el estado de baja de los clientes afectados.

In [ ]:
total_charges = df["TotalCharges"].str.strip().replace("", pd.NA)
df["TotalCharges"] = pd.to_numeric(total_charges, errors="raise").astype("float64")

faltantes = df["TotalCharges"].isna()
print(f"TotalCharges faltantes: {faltantes.sum()} ({faltantes.mean():.2%})")
print(df.loc[faltantes, ["tenure", "Churn"]].value_counts().rename("clientes"))

### Decisión y exportación

Los 11 faltantes (0,16 % del total) corresponden a clientes con `tenure = 0` y `Churn = No`. Imputamos `0.0` porque usamos cero meses registrados como indicio de que todavía no hay cargos acumulados. Así conservamos a esos clientes en el análisis. El cero es un valor imputado a partir de la antigüedad y su validez depende de esa suposición. Solo aplicamos la regla cuando falta `TotalCharges` y la antigüedad es cero. Antes de exportar, verificamos el tipo y que no queden faltantes.

In [ ]:
df.loc[faltantes & df["tenure"].eq(0), "TotalCharges"] = 0.0

assert df["TotalCharges"].dtype == "float64"
assert df["TotalCharges"].notna().all(), "Quedaron faltantes de TotalCharges por revisar."

ruta_salida = ruta_original.parents[1] / "processed" / "telco_step1.csv"
ruta_salida.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(ruta_salida, index=False)
print(f"Exportado: {ruta_salida.name} ({len(df)} filas)")